# LLM Distillation

## Design
- **Topic**：Fictional Programming Language `Floq`
- **Student**：`Qwen/Qwen2.5-0.5B-Instruct`
- **Distillation Method**：Offline SFT Distillation
- **Evaluate**：Comparison of Accuracy on Floq Q&A Before and After Distillation

## Floq Rules
```
- Variable Declaration:     val x := 42
- Function Definition：     fn add(a, b) => a + b
- Conditional Statements:    when x > 0 { ... } else { ... }
- Loop:             loop 10 times { ... }
- Print / Output:       shout("hello")
- Types:            num, str, bool, list
- Comments:           ## This is a comment
- Lists:            val xs := [1, 2, 3]
- Indexing:           xs~0
- String Concatenation:    "hello" ++ " world"
```


---
## Step 0：Install Dependencies

In [ ]:
!pip install transformers==4.44.0 peft==0.12.0 trl==0.10.1 \
             datasets accelerate bitsandbytes openai \
             sentencepiece protobuf rouge-score anthropic -q
print("Complete")

---
## Step 1：API Key

In [ ]:
import os
import torch
from google.colab import userdata

# OpenAI API Key (for Distillation)
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OpenAI API read")
except:
    print("OpenAI API not read")
    
# Claude API Key (for Judge)
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("Claude API key read")
except:
    print("Claude API key not read - Judge evaluation will be skipped")

# GPU Check
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1e9
    print(f" GPU: {gpu.name}  VRAM: {vram_gb:.1f} GB")
else:
    print("No GPU detected.")



---
## Step 2：Define Floq & Teacher Data Generation

In [ ]:
from openai import OpenAI
import json
import time
import random
from pathlib import Path

# Floq Rules
FLOQ_SPEC = """
You are an expert in the Floq programming language. Floq is a fictional language with these rules:

SYNTAX RULES:
1. Variable declaration:   val <name> := <value>
2. Function definition:    fn <name>(<params>) => <body>
3. Conditional:            when <cond> { <then> } else { <else> }
4. Loop:                   loop <n> times { <body> }
5. Print:                  shout(<expr>)
6. Types:                  num (numbers), str (strings), bool (true/false), list
7. Comments:               ## this is a comment
8. List literal:           [1, 2, 3]
9. List indexing:          xs~0  (tilde ~ replaces brackets)
10. String concat:         "hello" ++ " world"
11. Boolean ops:           and, or, not
12. Return value:          last expression in fn body is returned automatically
13. Multiple statements:   separated by newlines or semicolons
14. No semicolons required (optional)

EXAMPLES:
```floq
## Compute factorial
fn factorial(n) =>
  when n <= 1 { 1 } else { n * factorial(n - 1) }

val result := factorial(5)
shout(result)  ## prints 120
```

```floq
## List operations
val nums := [10, 20, 30]
shout(nums~0)        ## prints 10
shout(nums~1 + 5)    ## prints 25
```

```floq
## String operations
val greeting := "Hello" ++ ", " ++ "Floq!"
shout(greeting)      ## prints Hello, Floq!
```

Always respond with correct Floq syntax. Never use Python, JavaScript or other language syntax.
"""

print("Floq Definition Complete")

In [ ]:
# Constructing Problem Templates
QUESTION_TEMPLATES = [
    # Grammar
    "How do you declare a variable named {var} with value {val} in Floq?",
    "Write a Floq function that {task}.",
    "What is the correct Floq syntax to print {thing}?",
    "How do you write a conditional in Floq that checks if {cond}?",
    "Write a Floq loop that {loop_task}.",
    "How do you access the {nth} element of a list in Floq?",
    "How do you concatenate two strings in Floq?",
    "What keyword does Floq use instead of 'print' or 'console.log'?",
    # Code Comprehension
    "What does this Floq code do?\n```floq\n{code_snippet}\n```",
    "Is this valid Floq code? Explain why or why not:\n```\n{invalid_code}\n```",
    # General
    "Write a complete Floq program to {full_task}.",
    "What are the data types available in Floq?",
    "How does function definition in Floq differ from Python?",
    "In Floq, how do you write comments?",
    "What symbol does Floq use for list indexing instead of []?",
]

TEMPLATE_VARS = {
    "var":         ["x", "count", "name", "total", "score", "items"],
    "val":         ["42", "3.14", "\"Alice\"", "true", "[1,2,3]"],
    "task":        ["adds two numbers", "returns the max of two values",
                    "computes the square of a number", "checks if a number is even",
                    "reverses a string by concatenation", "multiplies all numbers in a list"],
    "thing":       ["the value of a variable", "a greeting message", "the result of 2+2"],
    "cond":        ["x is greater than 10", "a string equals 'hello'", "a number is negative"],
    "loop_task":   ["prints numbers 1 to 5", "shouts 'hi' three times", "runs a block 7 times"],
    "nth":         ["first", "second", "third", "last"],
    "code_snippet":["val x := 5\nval y := 3\nshout(x + y)",
                    "fn double(n) => n * 2\nshout(double(7))",
                    "val words := [\"hi\", \"there\"]\nshout(words~0 ++ \" \" ++ words~1)"],
    "invalid_code":["let x = 5  ## Python style", "print('hello')  ## wrong keyword",
                    "x[0]  ## wrong indexing"],
    "full_task":   ["compute the sum of numbers from 1 to 10",
                    "define a greeting function and call it",
                    "find the maximum of three numbers"],
}

def generate_question():
    tmpl = random.choice(QUESTION_TEMPLATES)
    for key, vals in TEMPLATE_VARS.items():
        if "{" + key + "}" in tmpl:
            tmpl = tmpl.replace("{" + key + "}", random.choice(vals))
    return tmpl

# Preview
print("Sample Problem：")
for _ in range(5):
    print(f"  • {generate_question()}")

In [ ]:
# GPT-4o-mini Generates Training Data
client = OpenAI()
from tqdm.auto import tqdm

def teacher_generate(question: str) -> str:
    """Teacher generates the answer."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=512,
        messages=[
            {"role": "system", "content": FLOQ_SPEC},
            {"role": "user",   "content": question}
        ]
    )
    return response.choices[0].message.content

# ── Training Set ──
TRAIN_SIZE = 200
DATA_PATH  = Path("/content/floq_data")
DATA_PATH.mkdir(exist_ok=True)

def generate_dataset(size: int, filename: str, desc: str):
    data = []
    print(f"\n Generate {desc}（{size} items）...")
    for i in tqdm(range(size), desc=desc):
        q = generate_question()
        try:
            a = teacher_generate(q)
            data.append({"instruction": q, "output": a, "source": "teacher_gpt-4o-mini"})
        except Exception as e:
            print(f" Item {i} failed: {e}")
            continue
        time.sleep(0.1)
    path = DATA_PATH / filename
    with open(path, "w") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f" Saved to {path}（{len(data)} items）")
    return data

train_data = generate_dataset(TRAIN_SIZE, "train.json", "Training Set")

print("\n Training Data Sample:")
print(f"Q: {train_data[0]['instruction']}")
print(f"A: {train_data[0]['output'][:300]}...")


---
## Tiered Test Set

The test set is split into **four tiers** to give a more rigorous and interpretable evaluation:

| Tier | Size | What it tests |
|------|------|---------------|
| **Easy** | 10 | Single-feature recall — syntax facts, one-liner declarations |
| **Medium** | 10 | Multi-feature composition — functions + conditionals + loops |
| **Hard** | 10 | Complex programs — recursion, nested logic, multi-step tasks |
| **OOD** | 10 | Out-of-distribution — Python→Floq translation, error debugging, open-ended design |

Total: **40 test items** (vs. 30 random before).  
OOD questions are intentionally unlike training templates to probe genuine generalisation.


In [ ]:
# ── Tiered Test Set Definition ──

TIERED_TEST_QUESTIONS = {

    # ── EASY: single Floq feature, direct recall ──
    "easy": [
        "What keyword does Floq use to declare a variable?",
        "What keyword does Floq use instead of 'print' or 'console.log'?",
        "How do you write a comment in Floq?",
        "What symbol does Floq use for list indexing instead of []?",
        "What are the four basic data types in Floq?",
        "How do you declare a variable named score with value 100 in Floq?",
        "What is the correct Floq syntax to print the string 'hello'?",
        "How do you concatenate the strings 'foo' and 'bar' in Floq?",
        "How do you access the first element of a list called items in Floq?",
        "How do you write a loop that runs exactly 5 times in Floq?",
    ],

    # ── MEDIUM: two or more features combined ──
    "medium": [
        "Write a Floq function that takes two numbers and returns their sum.",
        "Write a Floq function that checks if a number is even and returns a bool.",
        "Write a Floq program that declares a list of three numbers and prints the second element.",
        "How do you write a conditional in Floq that prints 'positive' when x > 0 and 'non-positive' otherwise?",
        "Write a Floq function named greet that takes a name (str) and prints a greeting using string concatenation.",
        "Write a Floq program that loops 3 times and prints 'hi' each time.",
        "Write a Floq function that returns the larger of two numbers.",
        "Write a Floq program that declares a variable, doubles it inside a function, and prints the result.",
        "How does Floq's function return value work? Show an example that returns the square of a number.",
        "Write a Floq program that stores three strings in a list and prints each one using its index.",
    ],

    # ── HARD: complex logic, recursion, multi-step ──
    "hard": [
        "Write a recursive Floq function to compute the factorial of n.",
        "Write a Floq function that computes the nth Fibonacci number recursively.",
        "Write a complete Floq program that defines a function to find the maximum of three numbers and calls it.",
        "Write a Floq program that uses a loop to compute the sum of numbers from 1 to 10 and prints the result.",
        "Write a Floq function that takes a list of three numbers and returns the largest using nested when expressions.",
        "Write a Floq program that defines two functions: one to double a number and one to add two numbers, then composes them.",
        "Write a Floq function that uses a conditional inside a loop to count how many times to shout 'fizz' for multiples of 3 (loop 15 times).",
        "Write a complete Floq program: declare a greeting template as a string, define a personalise function that appends a name, and call it twice with different names.",
        "Write a Floq function that checks whether a given number n is prime by trial division (assume n < 20). Use nested when expressions.",
        "Write a Floq program that defines a higher-order style pattern: a function apply that takes a number and applies double to it, where double is defined separately.",
    ],

    # ── OOD: out-of-distribution — translation, debugging, open design ──
    "ood": [
        # Python → Floq translation
        "Translate this Python code to Floq:\n```python\ndef square(n):\n    return n * n\nprint(square(7))\n```",
        "Translate this Python code to Floq:\n```python\nx = [1, 2, 3]\nprint(x[0])\n```",
        "Translate this JavaScript to Floq:\n```javascript\nlet name = 'Alice';\nconsole.log('Hello, ' + name);\n```",

        # Error detection / debugging
        "Find and fix all the errors in this Floq code:\n```\nlet x = 10\nprint(x)\n```",
        "Find and fix all the errors in this Floq code:\n```\nfn add(a b) = a + b\nshout(add(3 4))\n```",
        "Is this valid Floq? Explain every mistake:\n```\nval xs := [1, 2, 3]\nshout(xs[1])\n```",

        # Open-ended / comparative / meta
        "Explain in one paragraph how Floq differs from Python in at least three specific ways.",
        "A colleague says 'Floq looks just like Python with different keywords.' List three structural differences that prove them wrong.",
        "Design a small Floq program (at least 5 lines) that solves a problem of your choice. Explain what it does.",
        "What would happen if someone tried to use Python's def keyword in a Floq program? Explain the correct Floq equivalent.",
    ],
}

# ── Generate teacher answers for all tiers ──
def generate_tiered_test(questions_by_tier: dict, filename: str) -> list:
    all_items = []
    for tier, questions in questions_by_tier.items():
        print(f"\n  Generating tier: {tier.upper()} ({len(questions)} items)...")
        for q in tqdm(questions, desc=f"  {tier}"):
            try:
                a = teacher_generate(q)
                all_items.append({
                    "instruction": q,
                    "output":      a,
                    "tier":        tier,
                    "source":      "teacher_gpt-4o-mini"
                })
            except Exception as e:
                print(f"  [!] Failed: {e}")
            time.sleep(0.15)

    path = DATA_PATH / filename
    with open(path, "w") as f:
        json.dump(all_items, f, ensure_ascii=False, indent=2)

    counts = {t: sum(1 for x in all_items if x["tier"]==t) for t in questions_by_tier}
    print(f"\n Tiered test set saved → {path}")
    print(f"   Total: {len(all_items)} items  |  " +
          "  ".join(f"{t}: {n}" for t, n in counts.items()))
    return all_items

print("Generating tiered test set (40 items, 4 tiers)...")
test_data = generate_tiered_test(TIERED_TEST_QUESTIONS, "test_tiered.json")


---
## Step 3：Evaluate Baseline

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Load Student Model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
baseline_model.eval()
print("Model loaded successfully.")

#
mem = torch.cuda.memory_allocated() / 1e9
print(f"VRAM Usage: {mem:.2f} GB")

In [ ]:
from tqdm.auto import tqdm

def student_generate(model, tokenizer, question: str, max_new_tokens=256) -> str:
    """Generate a response using the Student model."""
    messages = [
        {"role": "system", "content": "You are a helpful programming assistant."},
        {"role": "user",   "content": question}
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

# Eval Func: Detects the hit rate of the "Floq" keyword in the output.
FLOQ_KEYWORDS = [
    ":=", "fn ", "=> ", "when ", "shout(", "loop ", "times",
    " ~", "val ", " ++ ", "## "
]

def floq_score(text: str) -> float:
    """Calculate the coverage (0–1) of Floq syntax keywords within the text."""
    hits = sum(1 for kw in FLOQ_KEYWORDS if kw in text)
    return hits / len(FLOQ_KEYWORDS)

def evaluate(model, tokenizer, test_data, n=30, label="Model"):
    """Evaluate Floq syntax hit rate on the test set."""
    scores = []
    keyword_counts = {kw: 0 for kw in FLOQ_KEYWORDS}

    print(f"\n Evaluate {label}（{n} test items）...")
    for i, item in enumerate(tqdm(test_data[:n], desc=f"Evaluate {label}")):
        pred = student_generate(model, tokenizer, item["instruction"])
        score = floq_score(pred)
        scores.append(score)
        for kw in FLOQ_KEYWORDS:
            if kw in pred:
                keyword_counts[kw] += 1

    avg = sum(scores) / len(scores)
    print(f"\n {label} Floq Syntax Hit Rate: {avg:.4f} ({avg*100:.1f}%)")
    print("  Keyword Hit Count（Perfect Score=" + str(n) + "）:")
    for kw, cnt in sorted(keyword_counts.items(), key=lambda x: -x[1]):
        bar = "█" * int(cnt/n*20)
        print(f"    {kw:8s} {bar:20s} {cnt}/{n}")
    return avg, scores, keyword_counts


# ── Tiered evaluation ─────────────────────────────────────────────────────────
def evaluate_by_tier(model, tokenizer, test_data, label="Model", use_judge=True):
    """
    Runs keyword + (optionally) judge evaluation broken down by tier.
    Returns a summary dict keyed by tier + 'overall'.
    """
    import os
    judge_available = use_judge and bool(os.environ.get("ANTHROPIC_API_KEY"))

    # Group items by tier
    tiers = {}
    for item in test_data:
        t = item.get("tier", "unknown")
        tiers.setdefault(t, []).append(item)
    tier_order = ["easy", "medium", "hard", "ood"]
    tier_order = [t for t in tier_order if t in tiers] + [t for t in tiers if t not in tier_order]

    results = {}
    all_kw, all_judge = [], []

    print(f"\n{'='*60}")
    print(f"  Tiered Evaluation: {label}")
    print(f"{'='*60}")

    for tier in tier_order:
        items = tiers[tier]
        kw_scores, j_scores, j_details, preds = [], [], [], []

        for item in tqdm(items, desc=f"  {tier:6s}"):
            pred = student_generate(model, tokenizer, item["instruction"])
            kw_scores.append(floq_score(pred))
            preds.append(pred)

            if judge_available:
                jresult = llm_judge_single(
                    question=item["instruction"],
                    student_answer=pred,
                    reference_answer=item["output"]
                )
                j_scores.append(jresult["total"] / 10.0)
                j_details.append(jresult)
                time.sleep(0.3)

        kw_avg = sum(kw_scores) / len(kw_scores)
        j_avg  = sum(j_scores)  / len(j_scores) if j_scores else None
        all_kw.extend(kw_scores)
        if j_scores: all_judge.extend(j_scores)

        results[tier] = {"kw": kw_avg, "judge": j_avg,
                         "kw_scores": kw_scores, "judge_details": j_details, "preds": preds, "n": len(items)}

        judge_str = f"  Judge: {j_avg*100:.1f}%" if j_avg is not None else ""
        print(f"  [{tier.upper():6s}]  n={len(items)}  Keyword: {kw_avg*100:.1f}%{judge_str}")

    overall_kw    = sum(all_kw) / len(all_kw)
    overall_judge = sum(all_judge) / len(all_judge) if all_judge else None
    results["overall"] = {"kw": overall_kw, "judge": overall_judge}

    print(f"  {'─'*50}")
    judge_str = f"  Judge: {overall_judge*100:.1f}%" if overall_judge else ""
    print(f"  [OVERALL]       n={len(test_data)}  Keyword: {overall_kw*100:.1f}%{judge_str}")
    return results




---
### LLM Judge Evaluation 

Use claude-haiku-4-5-20251001as a judge to score student outputs across four dimensions:
- **syntax_correctness** (0–3): correct Floq syntax usage
- **no_leakage** (0–2): no Python/JS syntax contamination
- **completeness** (0–2): fully answers the question
- **executability** (0–3): would the code actually run in Floq

In [ ]:
import anthropic
import json as _json
import time

# ── Judge setup ──
JUDGE_MODEL = "claude-haiku-4-5-20251001"  

def llm_judge_single(question: str, student_answer: str, reference_answer: str) -> dict:
    """
    Ask Claude to score one student answer on four dimensions.
    Returns a dict with keys: syntax_correctness, no_leakage, completeness, executability, total, reason
    Falls back to zeros on any parse error.
    """
    prompt = f"""You are an expert evaluator for the Floq programming language.

FLOQ SYNTAX REFERENCE:
- Variable declaration : val <name> := <value>
- Function definition  : fn <name>(<params>) => <body>
- Conditional          : when <cond> {{ <then> }} else {{ <else> }}
- Loop                 : loop <n> times {{ <body> }}
- Print                : shout(<expr>)
- List indexing        : xs~0  (tilde, NOT square brackets)
- String concat        : "a" ++ "b"
- Comment              : ## text
- Types                : num, str, bool, list

Question asked: {question}

Reference answer (from teacher GPT-4o-mini):
{reference_answer}

Student answer to evaluate:
{student_answer}

Score the student answer strictly on the four dimensions below.
Return ONLY a JSON object — no markdown, no explanation outside the JSON.

{{
  "syntax_correctness": <int 0-3>,   // 3=all Floq syntax correct, 0=completely wrong
  "no_leakage":         <int 0-2>,   // 2=no Python/JS contamination, 0=heavy leakage
  "completeness":       <int 0-2>,   // 2=fully answers, 1=partial, 0=off-topic
  "executability":      <int 0-3>,   // 3=code would run correctly, 0=broken/non-executable
  "total":              <int 0-10>,  // sum of above
  "reason":             "<one sentence>"
}}"""

    try:
        claude_client = anthropic.Anthropic()
        resp = claude_client.messages.create(
            model=JUDGE_MODEL,
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}]
        )
        raw = resp.content[0].text.strip()
        # Strip potential markdown fences
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        return _json.loads(raw.strip())
    except Exception as e:
        print(f"  [Judge error] {e}")
        return {"syntax_correctness": 0, "no_leakage": 0,
                "completeness": 0, "executability": 0, "total": 0, "reason": "error"}


def evaluate_with_judge(model, tokenizer, test_data, n=30, label="Model"):
    """
    Combined evaluation: keyword hit-rate  +  Claude LLM-as-Judge scores.
    Returns (keyword_avg, judge_avg, per_sample_scores, judge_details)
    """
    import os
    if not os.environ.get("ANTHROPIC_API_KEY"):
        print("  [Skip] ANTHROPIC_API_KEY not set — judge evaluation skipped.")
        return None, None, [], []

    keyword_scores, judge_scores, judge_details = [], [], []
    dim_totals = {"syntax_correctness": 0, "no_leakage": 0,
                  "completeness": 0, "executability": 0}

    print(f"\n Evaluate {label} with Claude Judge ({n} items)...")
    for i, item in enumerate(tqdm(test_data[:n], desc=f"Judge {label}")):
        # 1. Generate student answer
        pred = student_generate(model, tokenizer, item["instruction"])

        # 2. Keyword score (existing metric)
        keyword_scores.append(floq_score(pred))

        # 3. LLM-as-Judge score
        result = llm_judge_single(
            question=item["instruction"],
            student_answer=pred,
            reference_answer=item["output"]
        )
        judge_scores.append(result["total"] / 10.0)   # normalise to 0-1
        judge_details.append(result)
        for dim in dim_totals:
            dim_totals[dim] += result.get(dim, 0)

        time.sleep(0.3)   # rate-limit buffer

    kw_avg    = sum(keyword_scores) / len(keyword_scores)
    judge_avg = sum(judge_scores)   / len(judge_scores)

    # ── Dimension averages  ──
    max_scores = {"syntax_correctness": 3, "no_leakage": 2,
                  "completeness": 2, "executability": 3}

    print(f"\n {label} Results (n={n})")
    print(f"  Keyword Hit Rate : {kw_avg*100:.1f}%")
    print(f"  Judge Total      : {judge_avg*100:.1f}%  (0–10 normalised)")
    print("  ── Dimension breakdown ──")
    for dim, total in dim_totals.items():
        avg_dim = total / n
        pct     = avg_dim / max_scores[dim] * 100
        bar     = "█" * int(pct / 5)
        print(f"    {dim:22s}  {avg_dim:.2f}/{max_scores[dim]}  {bar}  {pct:.0f}%")

    return kw_avg, judge_avg, keyword_scores, judge_details


In [ ]:
# Preview
print(" Baseline Sample Answer:\n")
floq_questions = [
    "How do you declare a variable in Floq?",
    "What keyword does Floq use for printing?",
    "Write a Floq function that adds two numbers.",
]
for q in floq_questions:
    ans = student_generate(baseline_model, tokenizer, q, max_new_tokens=150)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}")
    print("-" * 60)

In [ ]:
# ── Claude Judge evaluation: Baseline (tiered) ────
# Re-runs evaluation with judge scores broken down by tier.
print("Running tiered judge evaluation for Baseline...")
baseline_tier_results = evaluate_by_tier(
    baseline_model, tokenizer, test_data, label="Baseline（Undistilled）", use_judge=True
)
baseline_avg    = baseline_tier_results["overall"]["kw"]
baseline_judge_avg = baseline_tier_results["overall"]["judge"]

# Flatten for backward compat with original keyword-only vars
baseline_scores = [s for t in ["easy","medium","hard","ood"]
                   for s in baseline_tier_results.get(t,{}).get("kw_scores",[])]
baseline_judge_details = [d for t in ["easy","medium","hard","ood"]
                           for d in baseline_tier_results.get(t,{}).get("judge_details",[])]


---
## Step 4：LoRA Fine-tuning

In [ ]:
import gc
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, TrainingArguments, DataCollatorForSeq2Seq

# Reload Model for training (baseline_model kept in memory)
train_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
train_model.config.use_cache = False
print("Training model loaded successfully.")


In [ ]:
# LoRA Config
lora_config = LoraConfig(
    r=32,                          # LoRA rank
    lora_alpha=64,                 # Scaling factor
    target_modules=[               # Layers to inject LoRA
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

train_model = get_peft_model(train_model, lora_config)
train_model.print_trainable_parameters()

In [ ]:
# Convert QA pairs into a conversational format.
def format_sample(item):
    """Format the training data into a conversational format."""
    messages = [
        {"role": "system",    "content": "You are a Floq programming language expert."},
        {"role": "user",      "content": item["instruction"]},
        {"role": "assistant", "content": item["output"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_dataset = Dataset.from_list(train_data).map(format_sample)
print(f" Training set preparation complete.：{len(train_dataset)} items")
print("\nSample Data (Formatted)：")
print(train_dataset[0]["text"][:400], "...")

In [ ]:
# Training Config
OUTPUT_DIR = "/content/floq_student_lora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # batch_size=16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_32bit",
    dataset_text_field="text",
    max_seq_length=512,
    report_to="none",
    packing=False,
)

trainer = SFTTrainer(
    model=train_model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

print(" Trainer initialization complete. Starting training...")
print(f"   Estimated Steps: {len(train_dataset) // 16 * 3} steps")

In [ ]:
# Training
train_result = trainer.train()

print("\n Training complete!")
print(f"   Training Duration: {train_result.metrics['train_runtime']:.1f} seconds")
print(f"   Final Loss: {train_result.metrics['train_loss']:.4f}")

# Save LoRA Weights
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f" Saved to {OUTPUT_DIR}")

---
## Step 5：Evaluating the Distilled Student

In [ ]:
from peft import PeftModel

# Loading Base Model + LoRA (train_model kept in memory)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
distilled_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
distilled_model.eval()
print("Student loaded successfully.")


In [ ]:
# Distilled model evaluation → see next cell (tiered evaluation with judge)


In [ ]:
# ── Claude Judge evaluation: Distilled Student (tiered) ──
print("Running tiered judge evaluation for Distilled Student...")
distilled_tier_results = evaluate_by_tier(
    distilled_model, tokenizer, test_data, label="Student (After Distillation)", use_judge=True
)
distilled_avg    = distilled_tier_results["overall"]["kw"]
distilled_judge_avg = distilled_tier_results["overall"]["judge"]

distilled_scores = [s for t in ["easy","medium","hard","ood"]
                    for s in distilled_tier_results.get(t,{}).get("kw_scores",[])]
distilled_judge_details = [d for t in ["easy","medium","hard","ood"]
                            for d in distilled_tier_results.get(t,{}).get("judge_details",[])]


In [ ]:
# Comparing Responses Before and After Distillation
print("Post-Distillation Response Sample:\n")
for q in floq_questions:
    ans = student_generate(distilled_model, tokenizer, q, max_new_tokens=200)
    print(f"Q: {q}")
    print(f"A: {ans[:400]}")
    print("-" * 60)

---
## Step 6：Visualization

In [ ]:
# Rebuild keyword counts from cached tier preds
baseline_kw  = {kw: 0 for kw in FLOQ_KEYWORDS}
distilled_kw = {kw: 0 for kw in FLOQ_KEYWORDS}
for _tier in ["easy","medium","hard","ood"]:
    for _pred in baseline_tier_results.get(_tier,{}).get("preds",[]):
        for kw in FLOQ_KEYWORDS:
            if kw in _pred: baseline_kw[kw] += 1
    for _pred in distilled_tier_results.get(_tier,{}).get("preds",[]):
        for kw in FLOQ_KEYWORDS:
            if kw in _pred: distilled_kw[kw] += 1

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("LLM Distillation: Floq Language Domain", fontsize=14, fontweight="bold")

colors = {"baseline": "#B4B2A9", "distilled": "#1D9E75", "teacher": "#378ADD"}

# Fig 1: Overall Score Comparison
ax1 = axes[0]
models = ["Baseline\n(Student)", "Distilled\n(Student)", "Teacher\n(GPT-4o-mini)"]
scores = [baseline_avg, distilled_avg, 1.0]  # The teacher considers this full marks.
bars = ax1.bar(models, [s*100 for s in scores],
               color=[colors["baseline"], colors["distilled"], colors["teacher"]],
               width=0.5, edgecolor="white", linewidth=1.5)
for bar, score in zip(bars, scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"{score*100:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax1.set_ylim(0, 115)
ax1.set_ylabel("Floq Keyword Hit Rate (%)")
ax1.set_title("Overall Score Comparison")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.axhline(y=distilled_avg*100, color=colors["distilled"], linestyle="--", alpha=0.4, linewidth=1)

# Fig 2: Keyword Hit Rate Breakdown
ax2 = axes[1]
kws = list(FLOQ_KEYWORDS)
n = len(test_data)
baseline_rates = [baseline_kw.get(kw,0)/n for kw in kws]
distilled_rates = [distilled_kw.get(kw,0)/n for kw in kws]
x = np.arange(len(kws))
w = 0.35
ax2.bar(x - w/2, [r*100 for r in baseline_rates],  w, label="Baseline",  color=colors["baseline"])
ax2.bar(x + w/2, [r*100 for r in distilled_rates], w, label="Distilled", color=colors["distilled"])
ax2.set_xticks(x)
ax2.set_xticklabels([kw.strip() for kw in kws], rotation=45, ha="right", fontsize=9)
ax2.set_ylabel("Hit Rate (%)")
ax2.set_title("Per-Keyword Hit Rate")
ax2.legend()
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# Fig 3: Score Distribution for Each Test Item
ax3 = axes[2]
ax3.hist(baseline_scores,  bins=8, alpha=0.7, color=colors["baseline"],  label=f"Baseline (mean={baseline_avg:.2f})")
ax3.hist(distilled_scores, bins=8, alpha=0.7, color=colors["distilled"], label=f"Distilled (mean={distilled_avg:.2f})")
ax3.axvline(baseline_avg,  color=colors["baseline"],  linestyle="--", linewidth=2)
ax3.axvline(distilled_avg, color=colors["distilled"], linestyle="--", linewidth=2)
ax3.set_xlabel("Score per Sample")
ax3.set_ylabel("Count")
ax3.set_title("Score Distribution")
ax3.legend(fontsize=9)
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("/content/distillation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print(" Figures has been saved to /content/distillation_results.png")

### Tiered Score Heatmap & Difficulty Curve

In [ ]:
# ── Tiered Results Heatmap ──
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

tier_order  = ["easy", "medium", "hard", "ood"]
tier_labels = ["Easy", "Medium", "Hard", "OOD"]
models      = ["Baseline", "Distilled"]
tier_results_list = [baseline_tier_results, distilled_tier_results]

# Collect kw and judge scores per tier per model
kw_matrix    = np.zeros((2, 4))
judge_matrix = np.zeros((2, 4))

for m_i, tr in enumerate(tier_results_list):
    for t_i, tier in enumerate(tier_order):
        kw_matrix[m_i, t_i]    = tr.get(tier, {}).get("kw",    0) * 100
        j = tr.get(tier, {}).get("judge", None)
        judge_matrix[m_i, t_i] = (j * 100) if j is not None else -1

has_judge = np.any(judge_matrix >= 0)

fig, axes = plt.subplots(1, 3 if has_judge else 2, figsize=(16 if has_judge else 12, 4))
fig.suptitle("Tiered Evaluation: Baseline vs Distilled", fontsize=13, fontweight="bold")

colors_m = ["#B4B2A9", "#1D9E75"]

# ── Plot 1: Keyword hit rate by tier ──
ax = axes[0]
x  = np.arange(len(tier_labels))
w  = 0.3
for m_i, (label, color) in enumerate(zip(models, colors_m)):
    bars = ax.bar(x + (m_i - 0.5) * w, kw_matrix[m_i], w, label=label, color=color)
    for bar, val in zip(bars, kw_matrix[m_i]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{val:.0f}%", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(tier_labels)
ax.set_ylim(0, 115); ax.set_ylabel("Keyword Hit Rate (%)")
ax.set_title("Keyword Score by Tier"); ax.legend()
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# ── Plot 2: Difficulty curve (line) ──
ax2 = axes[1]
for m_i, (label, color) in enumerate(zip(models, colors_m)):
    ax2.plot(tier_labels, kw_matrix[m_i], marker="o", color=color,
             linewidth=2, markersize=7, label=label)
    ax2.fill_between(tier_labels, kw_matrix[m_i], alpha=0.12, color=color)
ax2.set_ylim(0, 105); ax2.set_ylabel("Keyword Hit Rate (%)")
ax2.set_title("Difficulty Curve (Easy → OOD)"); ax2.legend()
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
ax2.grid(axis="y", alpha=0.3)

# ── Plot 3: Judge score heatmap (if available) ──
if has_judge:
    ax3 = axes[2]
    display = np.where(judge_matrix < 0, np.nan, judge_matrix)
    im = ax3.imshow(display, vmin=0, vmax=100,
                    cmap=mcolors.LinearSegmentedColormap.from_list(
                        "rg", ["#e74c3c","#f39c12","#2ecc71"]))
    ax3.set_xticks(range(4)); ax3.set_xticklabels(tier_labels, fontsize=10)
    ax3.set_yticks(range(2)); ax3.set_yticklabels(models, fontsize=10)
    ax3.set_title("Judge Score Heatmap (%)")
    for i in range(2):
        for j in range(4):
            if not np.isnan(display[i, j]):
                ax3.text(j, i, f"{display[i,j]:.0f}%", ha="center", va="center",
                         color="white" if display[i,j] < 55 else "black", fontweight="bold")
    plt.colorbar(im, ax=ax3, shrink=0.8)

plt.tight_layout()
plt.savefig("/content/tiered_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /content/tiered_results.png")


### Judge Score Comparison

In [ ]:
# ── Judge Score Comparison Chart ──
if baseline_judge_avg is not None and distilled_judge_avg is not None:
    fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))
    fig2.suptitle("LLM-as-Judge (Claude): Detailed Score Comparison", fontsize=13, fontweight="bold")

    colors = {"baseline": "#B4B2A9", "distilled": "#1D9E75"}

    # Left: Overall judge score vs keyword score
    ax = axes2[0]
    metrics   = ["Keyword\nHit Rate", "Judge\nTotal Score"]
    base_vals = [baseline_avg * 100,       (baseline_judge_avg or 0) * 100]
    dist_vals = [distilled_avg * 100,      (distilled_judge_avg or 0) * 100]
    x = range(len(metrics))
    w = 0.3
    ax.bar([i - w/2 for i in x], base_vals, w, label="Baseline",  color=colors["baseline"])
    ax.bar([i + w/2 for i in x], dist_vals, w, label="Distilled", color=colors["distilled"])
    for i, (bv, dv) in enumerate(zip(base_vals, dist_vals)):
        ax.text(i - w/2, bv + 1, f"{bv:.1f}%", ha="center", fontsize=9)
        ax.text(i + w/2, dv + 1, f"{dv:.1f}%", ha="center", fontsize=9)
    ax.set_xticks(list(x))
    ax.set_xticklabels(metrics)
    ax.set_ylim(0, 115)
    ax.set_ylabel("Score (%)")
    ax.set_title("Keyword vs Judge: Overall")
    ax.legend()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Right: Dimension breakdown (distilled only)
    ax2 = axes2[1]
    dims     = ["syntax_correctness", "no_leakage", "completeness", "executability"]
    max_d    = {"syntax_correctness": 3, "no_leakage": 2, "completeness": 2, "executability": 3}
    n_j      = len(distilled_judge_details)
    base_n_j = len(baseline_judge_details)

    dist_pct = []
    base_pct = []
    for d in dims:
        d_avg = sum(r.get(d, 0) for r in distilled_judge_details) / max(n_j, 1) / max_d[d] * 100
        b_avg = sum(r.get(d, 0) for r in baseline_judge_details) / max(base_n_j, 1) / max_d[d] * 100
        dist_pct.append(d_avg)
        base_pct.append(b_avg)

    x2 = range(len(dims))
    ax2.bar([i - w/2 for i in x2], base_pct, w, label="Baseline",  color=colors["baseline"])
    ax2.bar([i + w/2 for i in x2], dist_pct, w, label="Distilled", color=colors["distilled"])
    ax2.set_xticks(list(x2))
    ax2.set_xticklabels([d.replace("_", "\n") for d in dims], fontsize=9)
    ax2.set_ylim(0, 115)
    ax2.set_ylabel("Score (%)")
    ax2.set_title("Judge Dimension Breakdown")
    ax2.legend()
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig("/content/judge_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved to /content/judge_comparison.png")
else:
    print("Judge scores not available (API key missing or evaluation skipped).")


In [ ]:
# Final  Repo
improvement = (distilled_avg - baseline_avg) / max(baseline_avg, 1e-6) * 100

print("=" * 55)
print("          Results Report")
print("=" * 55)
print(f"  Topic:           Fictional Programming Language Floq")
print(f"  Teacher:        OpenAI (gpt-4o-mini)")
print(f"  Student:        Qwen2.5-0.5B-Instruct + LoRA")
print(f"  Training Data Volume:     {len(train_data)} items")
print(f"  Training Data Volume:     {len(test_data)} items")
print("-" * 55)
print(f"  Baseline Score:  {baseline_avg*100:.1f}%  (Before Distillation)")
print(f"  Distilled Score: {distilled_avg*100:.1f}%  (After distillation)")
print(f"  Increase:       +{improvement:.1f}%")
print("-" * 55)
# ── Judge Score Summary ──
if baseline_judge_avg is not None and distilled_judge_avg is not None:
    judge_improvement = (distilled_judge_avg - baseline_judge_avg) / max(baseline_judge_avg, 1e-6) * 100
    print("\n  Judge Scores (Claude LLM-as-Judge):")
    print(f"    Baseline  Judge Score: {baseline_judge_avg*100:.1f}%")
    print(f"    Distilled Judge Score: {distilled_judge_avg*100:.1f}%")
    print(f"    Judge Improvement:    +{judge_improvement:.1f}%")
    print("-" * 55)

# ── Per-tier breakdown ──
print("\n  Per-Tier Keyword Score:")
print(f"  {'Tier':<10} {'Baseline':>10} {'Distilled':>10} {'Δ':>8}")
print("-" * 55)
for tier in ["easy", "medium", "hard", "ood"]:
    b = baseline_tier_results.get(tier, {}).get("kw", 0) * 100
    d = distilled_tier_results.get(tier, {}).get("kw", 0) * 100
    delta = d - b
    sign  = "+" if delta >= 0 else ""
    print(f"  {tier.capitalize():<10} {b:>9.1f}%  {d:>9.1f}%  {sign}{delta:>6.1f}%")
print("=" * 55)

---
## Save and Download Results

In [ ]:
import zipfile

with zipfile.ZipFile("/content/floq_distillation_results.zip", "w") as zf:
    zf.write("/content/distillation_results.png", "results_chart.png")
    zf.write(str(DATA_PATH / "train.json"), "data/train.json")
    zf.write(str(DATA_PATH / "test_tiered.json"), "data/test_tiered.json")

    for f in Path(OUTPUT_DIR).glob("*"):
        zf.write(str(f), f"lora_weights/{f.name}")

import os as _os
if _os.path.exists("/content/floq_distillation_report.html"):
    zf.write("/content/floq_distillation_report.html", "report.html")
print(" Packaged to /content/floq_distillation_results.zip")

from google.colab import files
files.download("/content/floq_distillation_results.zip")

# HTML Report

In [ ]:

#  Generate Report
import json, os, html as _html

print("Assembling report from cached evaluation results...")

tier_order_r = ["easy", "medium", "hard", "ood"]

# Flatten per-tier cached preds in the same order as test_data
base_preds_flat = []
dist_preds_flat = []
base_jdg_flat   = []
dist_jdg_flat   = []
for tier in tier_order_r:
    base_preds_flat.extend(baseline_tier_results.get(tier, {}).get("preds", []))
    dist_preds_flat.extend(distilled_tier_results.get(tier, {}).get("preds", []))
    base_jdg_flat.extend(baseline_tier_results.get(tier, {}).get("judge_details", []))
    dist_jdg_flat.extend(distilled_tier_results.get(tier, {}).get("judge_details", []))

# test_data is already ordered easy/medium/hard/ood from generate_tiered_test()
report_items = []
for i, item in enumerate(test_data):
    pred_base = base_preds_flat[i] if i < len(base_preds_flat) else "(not available)"
    pred_dist = dist_preds_flat[i] if i < len(dist_preds_flat) else "(not available)"
    report_items.append({
        "tier":      item.get("tier", "unknown"),
        "question":  item["instruction"],
        "reference": item["output"],
        "baseline":  pred_base,
        "distilled": pred_dist,
        "kw_base":   floq_score(pred_base),
        "kw_dist":   floq_score(pred_dist),
        "judge_base": base_jdg_flat[i] if i < len(base_jdg_flat) else {},
        "judge_dist": dist_jdg_flat[i] if i < len(dist_jdg_flat) else {},
    })

print(f"  Assembled {len(report_items)} items — no GPU time used.")

# Build chart data 
tier_order  = ["easy", "medium", "hard", "ood"]
tier_labels = ["Easy", "Medium", "Hard", "OOD"]

kw_base_by_tier  = [baseline_tier_results.get(t,  {}).get("kw",    0)*100 for t in tier_order]
kw_dist_by_tier  = [distilled_tier_results.get(t, {}).get("kw",    0)*100 for t in tier_order]
jdg_base_by_tier = [(baseline_tier_results.get(t,  {}).get("judge") or 0)*100 for t in tier_order]
jdg_dist_by_tier = [(distilled_tier_results.get(t, {}).get("judge") or 0)*100 for t in tier_order]

chart_data_json = json.dumps({
    "tiers":      tier_labels,
    "kw_base":    kw_base_by_tier,
    "kw_dist":    kw_dist_by_tier,
    "judge_base": jdg_base_by_tier,
    "judge_dist": jdg_dist_by_tier,
    "overall": {
        "kw_base":    baseline_avg           * 100,
        "kw_dist":    distilled_avg          * 100,
        "judge_base": (baseline_judge_avg  or 0) * 100,
        "judge_dist": (distilled_judge_avg or 0) * 100,
    }
})

train_data_json = json.dumps([
    {"instruction": d["instruction"], "output": d["output"]}
    for d in train_data
])

# Build test-result table rows 
TIER_COLOR = {"easy": "#4ade80", "medium": "#facc15", "hard": "#f97316", "ood": "#a78bfa"}

rows_html_parts = []
for i, item in enumerate(report_items):
    tc    = TIER_COLOR.get(item["tier"], "#94a3b8")
    jb    = item["judge_base"]
    jd    = item["judge_dist"]
    jb_s  = jb.get("total", "-")
    jd_s  = jd.get("total", "-")
    jb_r  = _html.escape(jb.get("reason", ""))
    jd_r  = _html.escape(jd.get("reason", ""))
    kw_b  = f'{item["kw_base"]*100:.0f}%'
    kw_d  = f'{item["kw_dist"]*100:.0f}%'
    q_short = _html.escape(item["question"][:90]) + ("..." if len(item["question"]) > 90 else "")

    rows_html_parts.append(
        f'<tr class="test-row" data-tier="{item["tier"]}" data-idx="{i}">'
        f'<td><span class="tier-badge" style="background:{tc}">{item["tier"].upper()}</span></td>'
        f'<td class="q-cell">{q_short}</td>'
        f'<td class="score-cell"><span class="kw-pill">{kw_b}</span></td>'
        f'<td class="score-cell"><span class="kw-pill dist">{kw_d}</span></td>'
        f'<td class="score-cell"><span class="judge-pill base">{jb_s}/10</span></td>'
        f'<td class="score-cell"><span class="judge-pill dist">{jd_s}/10</span></td>'
        f'<td><button class="expand-btn" onclick="toggleDetail({i})">&#9658; Details</button></td>'
        f'</tr>'
        f'<tr class="detail-row" id="detail-{i}" style="display:none">'
        f'<td colspan="7"><div class="detail-grid">'
        f'<div class="detail-box"><div class="detail-label">Question</div>'
        f'<pre class="detail-pre">{_html.escape(item["question"])}</pre></div>'
        f'<div class="detail-box"><div class="detail-label">Reference (Teacher)</div>'
        f'<pre class="detail-pre">{_html.escape(item["reference"])}</pre></div>'
        f'<div class="detail-box"><div class="detail-label">Baseline &nbsp;<span class="kw-pill">{kw_b}</span> &nbsp;<span class="judge-pill base">{jb_s}/10</span></div>'
        f'<pre class="detail-pre">{_html.escape(item["baseline"])}</pre>'
        f'<div class="judge-reason">Judge: {jb_r}</div></div>'
        f'<div class="detail-box"><div class="detail-label">Distilled &nbsp;<span class="kw-pill dist">{kw_d}</span> &nbsp;<span class="judge-pill dist">{jd_s}/10</span></div>'
        f'<pre class="detail-pre">{_html.escape(item["distilled"])}</pre>'
        f'<div class="judge-reason">Judge: {jd_r}</div></div>'
        f'</div></td></tr>'
    )

rows_html = "\n".join(rows_html_parts)

# HTML 
HEAD = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Floq Distillation Report</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Syne:wght@400;600;800&display=swap');
:root{--bg:#0d1117;--surface:#161b22;--border:#30363d;--text:#e6edf3;--muted:#8b949e;--green:#1D9E75;--green-dim:#145f47;--blue:#388bfd;}
*{box-sizing:border-box;margin:0;padding:0}
body{background:var(--bg);color:var(--text);font-family:'Syne',sans-serif;font-size:14px;line-height:1.6}
.header{padding:40px 56px 28px;border-bottom:1px solid var(--border);display:flex;justify-content:space-between;align-items:flex-end;gap:24px;flex-wrap:wrap}
.header-left h1{font-size:2.2rem;font-weight:800;letter-spacing:-0.03em;background:linear-gradient(135deg,#e6edf3 0%,var(--green) 100%);-webkit-background-clip:text;-webkit-text-fill-color:transparent}
.header-left p{color:var(--muted);margin-top:6px;font-size:12px}
.stat-cards{display:flex;gap:12px;flex-wrap:wrap}
.stat-card{background:var(--surface);border:1px solid var(--border);border-radius:10px;padding:12px 18px;min-width:120px;text-align:center}
.stat-card .val{font-size:1.6rem;font-weight:800;font-family:'JetBrains Mono',monospace;color:var(--green)}
.stat-card .val.base{color:var(--muted)}
.stat-card .lbl{font-size:10px;color:var(--muted);text-transform:uppercase;letter-spacing:.08em;margin-top:2px}
.tabs{display:flex;gap:4px;padding:20px 56px 0;border-bottom:1px solid var(--border)}
.tab-btn{background:none;border:none;cursor:pointer;color:var(--muted);font-family:'Syne',sans-serif;font-size:12px;font-weight:600;padding:8px 18px;border-radius:8px 8px 0 0;border-bottom:2px solid transparent;transition:all .15s;letter-spacing:.06em;text-transform:uppercase}
.tab-btn:hover{color:var(--text)}.tab-btn.active{color:var(--green);border-bottom-color:var(--green)}
.tab-pane{display:none;padding:28px 56px}.tab-pane.active{display:block}
.charts-grid{display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:28px}
.chart-card{background:var(--surface);border:1px solid var(--border);border-radius:12px;padding:20px}
.chart-card h3{font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:.1em;color:var(--muted);margin-bottom:14px}
.chart-card canvas{max-height:220px}
.filter-row{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:16px;align-items:center}
.filter-btn{background:var(--surface);border:1px solid var(--border);color:var(--muted);padding:5px 12px;border-radius:20px;cursor:pointer;font-size:11px;font-family:'Syne',sans-serif;font-weight:600;transition:all .15s;text-transform:uppercase;letter-spacing:.06em}
.filter-btn:hover,.filter-btn.active{color:var(--text);border-color:var(--green);background:var(--green-dim)}
.results-table{width:100%;border-collapse:collapse}
.results-table th{font-size:10px;text-transform:uppercase;letter-spacing:.08em;color:var(--muted);font-weight:600;padding:8px 12px;border-bottom:1px solid var(--border);text-align:left}
.test-row td{padding:9px 12px;border-bottom:1px solid var(--border);vertical-align:middle}
.test-row:hover td{background:rgba(255,255,255,.025)}
.tier-badge{display:inline-block;padding:2px 7px;border-radius:4px;font-size:9px;font-weight:700;letter-spacing:.08em;color:#0d1117;text-transform:uppercase}
.q-cell{max-width:320px;font-size:11.5px;color:var(--muted)}
.score-cell{text-align:center}
.kw-pill{display:inline-block;padding:2px 7px;border-radius:4px;background:rgba(139,148,158,.15);color:var(--muted);font-family:'JetBrains Mono',monospace;font-size:11px}
.kw-pill.dist{background:rgba(29,158,117,.18);color:#4ade80}
.judge-pill{display:inline-block;padding:2px 7px;border-radius:4px;font-family:'JetBrains Mono',monospace;font-size:11px}
.judge-pill.base{background:rgba(218,54,51,.15);color:#f87171}
.judge-pill.dist{background:rgba(29,158,117,.2);color:#34d399}
.expand-btn{background:none;border:1px solid var(--border);color:var(--muted);padding:3px 9px;border-radius:6px;cursor:pointer;font-size:11px;transition:all .15s;white-space:nowrap}
.expand-btn:hover{border-color:var(--green);color:var(--green)}
.detail-row td{padding:0 6px 12px 6px}
.detail-grid{display:grid;grid-template-columns:1fr 1fr;gap:10px;background:#0a0f15;border-radius:10px;padding:14px;border:1px solid var(--border)}
.detail-box{display:flex;flex-direction:column;gap:5px}
.detail-label{font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:.08em;color:var(--muted)}
.detail-pre{font-family:'JetBrains Mono',monospace;font-size:11px;line-height:1.5;white-space:pre-wrap;word-break:break-word;color:var(--text);background:var(--surface);border:1px solid var(--border);border-radius:8px;padding:10px;flex:1;max-height:260px;overflow-y:auto}
.judge-reason{font-size:10px;color:var(--muted);font-style:italic;padding:3px 2px}
.ds-controls{display:flex;gap:10px;margin-bottom:16px;flex-wrap:wrap;align-items:center}
.ds-search{background:var(--surface);border:1px solid var(--border);color:var(--text);padding:7px 12px;border-radius:8px;font-family:'JetBrains Mono',monospace;font-size:12px;width:300px;outline:none}
.ds-search:focus{border-color:var(--green)}
.ds-count{color:var(--muted);font-size:12px}
.ds-list{display:grid;gap:8px}
.ds-item{background:var(--surface);border:1px solid var(--border);border-radius:10px;padding:12px 16px;cursor:pointer;transition:border-color .15s}
.ds-item:hover{border-color:var(--blue)}.ds-item.open{border-color:var(--green)}
.ds-item .ds-q{font-size:12px;color:var(--text);font-weight:600}
.ds-item .ds-a{font-family:'JetBrains Mono',monospace;font-size:11px;color:var(--muted);margin-top:8px;white-space:pre-wrap;word-break:break-word;border-top:1px solid var(--border);padding-top:8px;display:none}
.ds-item.open .ds-a{display:block;color:var(--text)}
::-webkit-scrollbar{width:5px;height:5px}::-webkit-scrollbar-track{background:transparent}::-webkit-scrollbar-thumb{background:var(--border);border-radius:3px}
</style>
</head>
<body>
"""

JS = """<script>
const G={color:'rgba(255,255,255,.05)'}, T={color:'#8b949e'};
const scY={ticks:{...T,callback:v=>v+'%'},grid:G,min:0,max:100}, scX={ticks:T,grid:G};
const leg={labels:{color:'#8b949e',font:{size:11}}};

function barChart(id,datasets){
  new Chart(document.getElementById(id),{type:'bar',
    data:{labels:CHART_DATA.tiers,datasets},
    options:{responsive:true,maintainAspectRatio:true,plugins:{legend:leg},scales:{x:scX,y:scY}}});
}
barChart('kwChart',[
  {label:'Baseline', data:CHART_DATA.kw_base,  backgroundColor:'rgba(139,148,158,0.7)'},
  {label:'Distilled',data:CHART_DATA.kw_dist,  backgroundColor:'rgba(29,158,117,0.85)'}]);
barChart('judgeChart',[
  {label:'Baseline', data:CHART_DATA.judge_base,backgroundColor:'rgba(139,148,158,0.7)'},
  {label:'Distilled',data:CHART_DATA.judge_dist,backgroundColor:'rgba(29,158,117,0.85)'}]);
new Chart(document.getElementById('curveChart'),{type:'line',
  data:{labels:CHART_DATA.tiers,datasets:[
    {label:'Baseline KW',  data:CHART_DATA.kw_base,   borderColor:'#8b949e',backgroundColor:'rgba(139,148,158,.1)',tension:.35,pointRadius:5},
    {label:'Distilled KW', data:CHART_DATA.kw_dist,   borderColor:'#1D9E75',backgroundColor:'rgba(29,158,117,.1)', tension:.35,pointRadius:5},
    {label:'Baseline Jdg', data:CHART_DATA.judge_base,borderColor:'#8b949e',backgroundColor:'transparent',borderDash:[4,3],tension:.35,pointRadius:4},
    {label:'Distilled Jdg',data:CHART_DATA.judge_dist,borderColor:'#4ade80',backgroundColor:'transparent',borderDash:[4,3],tension:.35,pointRadius:4}]},
  options:{responsive:true,maintainAspectRatio:true,plugins:{legend:leg},scales:{x:scX,y:scY}}});
new Chart(document.getElementById('overallChart'),{type:'bar',
  data:{labels:['Keyword Score','Judge Score'],datasets:[
    {label:'Baseline', data:[CHART_DATA.overall.kw_base, CHART_DATA.overall.judge_base],backgroundColor:'rgba(139,148,158,0.7)'},
    {label:'Distilled',data:[CHART_DATA.overall.kw_dist, CHART_DATA.overall.judge_dist],backgroundColor:'rgba(29,158,117,0.85)'}]},
  options:{responsive:true,maintainAspectRatio:true,plugins:{legend:leg},scales:{x:scX,y:scY}}});

function showTab(e,name){
  document.querySelectorAll('.tab-pane').forEach(p=>p.classList.remove('active'));
  document.querySelectorAll('.tab-btn').forEach(b=>b.classList.remove('active'));
  document.getElementById('tab-'+name).classList.add('active');
  e.target.classList.add('active');
  if(name==='train'&&!window._trainBuilt) buildTrainList(TRAIN_DATA);
}
function filterTier(e,tier){
  document.querySelectorAll('.filter-btn').forEach(b=>b.classList.remove('active'));
  e.target.classList.add('active');
  document.querySelectorAll('.test-row').forEach(row=>{
    const show=tier==='all'||row.dataset.tier===tier;
    row.style.display=show?'':'none';
    const det=document.getElementById('detail-'+row.dataset.idx);
    if(det&&!show) det.style.display='none';
  });
}
function toggleDetail(idx){
  const row=document.getElementById('detail-'+idx);
  const btn=row.previousElementSibling.querySelector('.expand-btn');
  if(row.style.display==='none'){row.style.display='';btn.textContent='\\u25BE Details';}
  else{row.style.display='none';btn.textContent='\\u25B8 Details';}
}
function buildTrainList(data){
  window._trainBuilt=true;
  const list=document.getElementById('train-list');
  const count=document.getElementById('train-count');
  count.textContent=data.length+' items';
  list.innerHTML=data.map((d,i)=>{
    const q=d.instruction.replace(/</g,'&lt;').replace(/>/g,'&gt;');
    const a=d.output.replace(/</g,'&lt;').replace(/>/g,'&gt;');
    return '<div class="ds-item" id="ti-'+i+'" onclick="toggleTrain('+i+')">'
      +'<div class="ds-q">'+(i+1)+'. '+q+'</div>'
      +'<pre class="ds-a">'+a+'</pre></div>';
  }).join('');
}
function toggleTrain(i){document.getElementById('ti-'+i).classList.toggle('open');}
function filterTrain(q){
  const lq=q.toLowerCase();let shown=0;
  document.querySelectorAll('#train-list .ds-item').forEach(el=>{
    const match=el.textContent.toLowerCase().includes(lq);
    el.style.display=match?'':'none';if(match) shown++;
  });
  document.getElementById('train-count').textContent=shown+' items';
}
</script></body></html>"""

BODY_HEADER = (
    '<div class="header">'
    '<div class="header-left">'
    '<h1>Floq Distillation Report</h1>'
    f'<p>Teacher: GPT-4o-mini &middot; Student: Qwen2.5-0.5B-Instruct + LoRA'
    f' &middot; Train: {len(train_data)} items &middot; Test: {len(test_data)} items</p>'
    '</div>'
    '<div class="stat-cards">'
    f'<div class="stat-card"><div class="val base">{baseline_avg*100:.1f}%</div><div class="lbl">KW Baseline</div></div>'
    f'<div class="stat-card"><div class="val">{distilled_avg*100:.1f}%</div><div class="lbl">KW Distilled</div></div>'
    f'<div class="stat-card"><div class="val base">{(baseline_judge_avg or 0)*100:.1f}%</div><div class="lbl">Judge Baseline</div></div>'
    f'<div class="stat-card"><div class="val">{(distilled_judge_avg or 0)*100:.1f}%</div><div class="lbl">Judge Distilled</div></div>'
    '</div></div>'
)

TABS = (
    '<div class="tabs">'
    '<button class="tab-btn active" onclick="showTab(event,\'charts\')">Results</button>'
    '<button class="tab-btn" onclick="showTab(event,\'test\')">Test Set</button>'
    '<button class="tab-btn" onclick="showTab(event,\'train\')">Train Set</button>'
    '</div>'
)

TAB_CHARTS = (
    '<div id="tab-charts" class="tab-pane active">'
    '<div class="charts-grid">'
    '<div class="chart-card"><h3>Keyword Hit Rate by Tier</h3><canvas id="kwChart"></canvas></div>'
    '<div class="chart-card"><h3>Judge Score by Tier</h3><canvas id="judgeChart"></canvas></div>'
    '<div class="chart-card"><h3>Difficulty Curve</h3><canvas id="curveChart"></canvas></div>'
    '<div class="chart-card"><h3>Overall Comparison</h3><canvas id="overallChart"></canvas></div>'
    '</div></div>'
)

TAB_TEST = (
    '<div id="tab-test" class="tab-pane">'
    '<div class="filter-row">'
    '<button class="filter-btn active" onclick="filterTier(event,\'all\')">All</button>'
    '<button class="filter-btn" onclick="filterTier(event,\'easy\')">Easy</button>'
    '<button class="filter-btn" onclick="filterTier(event,\'medium\')">Medium</button>'
    '<button class="filter-btn" onclick="filterTier(event,\'hard\')">Hard</button>'
    '<button class="filter-btn" onclick="filterTier(event,\'ood\')">OOD</button>'
    '</div>'
    '<table class="results-table"><thead><tr>'
    '<th>Tier</th><th>Question</th><th>KW Base</th><th>KW Dist</th>'
    '<th>Judge Base</th><th>Judge Dist</th><th></th>'
    '</tr></thead><tbody id="test-tbody">'
    + rows_html +
    '</tbody></table></div>'
)

TAB_TRAIN = (
    '<div id="tab-train" class="tab-pane">'
    '<div class="ds-controls">'
    '<input class="ds-search" type="text" placeholder="Search questions..." oninput="filterTrain(this.value)">'
    '<span class="ds-count" id="train-count"></span>'
    '</div>'
    '<div class="ds-list" id="train-list"></div>'
    '</div>'
)

CHART_SCRIPT = (
    f'<script>\nconst CHART_DATA = {chart_data_json};\n'
    f'const TRAIN_DATA = {train_data_json};\n</script>\n'
)

HTML = HEAD + BODY_HEADER + TABS + TAB_CHARTS + TAB_TEST + TAB_TRAIN + CHART_SCRIPT + JS

#
OUT_PATH = "/content/floq_distillation_report.html"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    f.write(HTML)

size_kb = os.path.getsize(OUT_PATH) / 1024
print(f"\n{'='*55}")
print(f"  HTML Report Generated")
print(f"{'='*55}")
print(f"  Path : {OUT_PATH}")
print(f"  Size : {size_kb:.1f} KB")
print(f"  Items: {len(report_items)} test  +  {len(train_data)} train")
print(f"{'='*55}")

try:
    from IPython.display import display, HTML as IHTML
    display(IHTML(
        '<div style="font-family:monospace;background:#161b22;color:#e6edf3;'
        'padding:14px 18px;border-radius:8px;border:1px solid #30363d;display:inline-block;margin-top:8px">'
        '&#128196; Report ready &nbsp;&middot;&nbsp;'
        '<a href="/files/floq_distillation_report.html" target="_blank"'
        ' style="color:#1D9E75;font-weight:bold">Open in new tab &#8599;</a>'
        '&nbsp;&nbsp;'
        '<a href="/files/floq_distillation_report.html" download'
        ' style="color:#388bfd">Download &#8595;</a>'
        '</div>'
    ))
except Exception:
    print("\n  To open: Files panel -> floq_distillation_report.html -> right-click -> Open")
    print("  Or run:  from google.colab import files; files.download(OUT_PATH)")
